#  Load and Explore the Dataset
---

## Objective
Get familiar with the dataset structure, count images per class, and visualise sample X-rays from both classes.

## Why this step matters
Before building any model, we need to understand:
- How many images are in each class — reveals **class imbalance**
- What the X-rays look like visually — builds **domain intuition**
- What image sizes and formats we are working with — informs **preprocessing decisions**

Skipping this step often leads to poor modelling decisions later.

## Dataset
We use the **Chest X-Ray Images (Pneumonia)** dataset from Kaggle (Kermany et al.).
It contains chest X-rays from paediatric patients split into:
- `train/` — model learns from this
- `val/` — model is tuned against this
- `test/` — final unseen evaluation only

Each split contains two folders: `NORMAL` and `PNEUMONIA`


---
## Step 1: Setup — Mount Drive and Import Libraries

**What:** Mount Google Drive to access the dataset and import all required libraries.

**Why:** The dataset is stored on Google Drive. Mounting gives Colab direct access without re-uploading. We import `os` for folder navigation, `PIL` for opening images, and `matplotlib` for visualisation.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/pneumonia-detection')
from src.config import TRAIN_DIR, VAL_DIR, TEST_DIR, CLASSES
from src.visualiser import plot_sample_images
print('Setup complete.')

---
## Step 2: Count Images Per Class

**What:** Count how many images exist in each class (NORMAL / PNEUMONIA) across all three splits.

**Why:** This tells us immediately whether the dataset is **balanced or imbalanced**. Class imbalance is one of the most common pitfalls in medical AI — if ignored, the model can appear accurate while actually being biased towards the majority class.


In [ ]:
print(f'{'Split':<8} {'NORMAL':>10} {'PNEUMONIA':>12} {'Total':>8}')
print('-' * 44)
for split, path in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    counts = {cls: len(os.listdir(os.path.join(path, cls))) for cls in CLASSES}
    total  = sum(counts.values())
    print(f'{split:<8} {counts["NORMAL"]:>10} {counts["PNEUMONIA"]:>12} {total:>8}')

---
## Step 3: Visualise Class Distribution

**What:** Plot bar charts showing the number of images per class for each split.

**Why:** A visual representation of class distribution makes imbalance immediately obvious. Bar charts are more intuitive than raw numbers when presenting to non-technical stakeholders like doctors or project supervisors.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Class Distribution Across Splits', fontsize=14, fontweight='bold')

for ax, (split, path) in zip(axes, [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]):
    counts = [len(os.listdir(os.path.join(path, cls))) for cls in CLASSES]
    bars   = ax.bar(CLASSES, counts, color=['#2196F3', '#F44336'], edgecolor='white')
    for bar, val in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 10, str(val),
                ha='center', fontweight='bold')
    ax.set_title(f'{split} Split')
    ax.set_ylabel('Number of Images')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## Step 4: Visualise Sample X-Ray Images

**What:** Display sample chest X-ray images from both NORMAL and PNEUMONIA classes.

**Why:** Visual inspection builds domain intuition. By looking at the images we can:
- Understand what features the model needs to detect
- Spot quality issues like blurry or mislabelled images
- Verify the dataset looks as expected before investing time in training


In [ ]:
plot_sample_images(TRAIN_DIR, n_per_class=5)

---
## Step 5: Check Image Properties

**What:** Inspect the size and format of sample images from the dataset.

**Why:** CNNs require **fixed-size inputs**. If images vary in size (which they do here), we must resize them during preprocessing. Knowing the size range helps us choose an appropriate target size — large enough to retain detail, small enough to be computationally feasible.


In [ ]:
import pandas as pd
shapes = []
for cls in CLASSES:
    folder = os.path.join(TRAIN_DIR, cls)
    files  = [f for f in os.listdir(folder)
               if f.lower().endswith(('.jpg','.jpeg','.png'))][:10]
    for fname in files:
        img = Image.open(os.path.join(folder, fname))
        shapes.append({'Class': cls, 'Size': img.size, 'Mode': img.mode})

df = pd.DataFrame(shapes)
print('Sample image sizes per class:')
print(df.groupby('Class')['Size'].apply(lambda x: x.value_counts().head(3)))
print('\nImage modes:', df['Mode'].unique())

---
## Observations and Key Findings

### Q&A

**1. What is the class distribution and imbalance in the training set?**
The training set contains 5216 images. 1341 (25.71%) are Normal and 3875 (74.29%) are Pneumonia — a significant class imbalance of approximately 3:1. This means a naive model that always predicts Pneumonia would achieve ~75% accuracy while being clinically useless.

**2. What are the visual differences between normal and pneumonia chest X-rays?**
Pneumonia X-rays typically display consolidations or infiltrates appearing as whitish areas within the lung fields, indicating inflammation or fluid accumulation. Normal X-rays present clear dark lung fields without such opacities.

**3. Why is image resizing necessary?**
CNNs require fixed-size inputs for their input layers. Varying image sizes would cause inconsistent data mapping and processing errors. We resize all images to 224x224 — large enough to retain clinical detail, standard for pretrained models like MobileNetV2.

### Key Findings
- Training set is heavily imbalanced: 74.29% Pneumonia vs 25.71% Normal
- Pneumonia X-rays show whitish consolidations in lung fields
- Normal X-rays show clear dark lung fields
- Images vary in size — standardisation to 224x224 is essential
- Validation set has only 16 images — too small for reliable validation metrics

### Next Steps
- Resize all images to 224x224 during preprocessing
- Address class imbalance during model training
- Use test set metrics as primary evaluation — not validation
